In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
from pyspark.sql import functions as F

# Retrieve values from widgets
ACCESS_KEY = "xxxxxxxxx"
SECRET_KEY = "xxxxxxxxx"
BUCKET_NAME =  "databricks98"

encoded_secret = SECRET_KEY.replace("/", "%2F") # Encodes slashes if they exist in your secret
s3_source_path = f"s3a://{ACCESS_KEY}:{encoded_secret}@{BUCKET_NAME}/"

In [0]:
import urllib.parse
# --- 2. WORKSPACE SETUP (The Catalog/Volume Fix) ---
# We are creating a dedicated 'schema' and 'volume' inside your 'workspace' catalog
# to store the checkpoints and the final table safely.
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.ckd_project")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.ckd_project.checkpoints")

# Correct path for Serverless Checkpoints (Avoiding the DBFS_DISABLED error)
checkpoint_path = "/Volumes/workspace/ckd_project/checkpoints/"

# Encode S3 keys for the direct URL handshake
encoded_secret = urllib.parse.quote_plus(SECRET_KEY)
s3_source_path = f"s3a://{ACCESS_KEY}:{encoded_secret}@{BUCKET_NAME}/"



✅ SUCCESS: Data loaded into workspace.ckd_project.ckd_bronze


bp_diastolic,bp_limit,sg,al,class,rbc,su,pc,pcc,ba,bgr,bu,sod,sc,pot,hemo,pcv,rbcc,wbcc,htn,dm,cad,appet,pe,ane,grf,stage,affected,age
null,discrete,null,null,discrete,discrete,null,discrete,discrete,discrete,null,null,null,null,null,null,null,null,null,discrete,discrete,discrete,discrete,discrete,discrete,null,discrete,null,null
null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
0.0,0,null,null,ckd,0,null,0,0,0,null,null,null,null,null,null,null,null,null,0,0,0,0,0,0,null,s1,1,null
0.0,0,null,null,ckd,0,null,0,0,0,null,null,null,null,null,null,null,null,null,0,0,0,0,0,0,null,s1,1,null
0.0,0,null,null,ckd,1,null,1,0,1,null,null,null,null,null,null,null,null,null,0,0,0,1,0,0,null,s1,1,null
1.0,1,null,null,ckd,0,null,0,0,0,null,null,null,null,null,null,null,null,null,0,0,0,0,0,0,null,s1,1,null
0.0,0,null,null,ckd,0,null,0,0,0,null,null,null,null,null,null,null,null,null,0,1,0,1,1,0,null,s1,1,null
1.0,1,null,null,notckd,0,null,0,0,0,null,null,null,null,null,null,null,null,null,0,0,0,0,0,0,null,s1,0,null
0.0,0,null,null,ckd,0,null,0,0,0,null,null,null,null,null,null,null,null,null,1,1,0,0,0,0,null,s1,1,null
0.0,0,null,null,ckd,0,null,0,0,0,null,null,null,null,null,null,null,null,null,0,0,0,0,0,0,null,s4,1,null


In [0]:
from pyspark.sql.functions import split, col
from pyspark.sql.types import StructType, StructField, StringType

# column list
column_names = [
    "bp_diastolic", "bp_limit", "sg", "al", "class", "rbc", "su", "pc", "pcc", "ba",
    "bgr", "bu", "sod", "sc", "pot", "hemo", "pcv", "rbcc", "wbcc", "htn", "dm", 
    "cad", "appet", "pe", "ane", "grf", "stage", "affected", "age"
]

# read raw as text (NO CSV parsing)
df_raw = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "text")
    .load(s3_source_path)
)

# split using COMMA (not tab) - the file is comma-delimited
df_split = df_raw.withColumn("cols", split(col("value"), ","))

# map columns safely by position
df_final = df_split.select(
    *[
        col("cols").getItem(i).alias(column_names[i])
        for i in range(len(column_names))
    ]
)

# remove header rows (first row has column names, second row has "discrete" values)
df_final = df_final.filter(
    (col("bp_diastolic") != "bp (Diastolic)") &
    (col("bp_diastolic") != "discrete") &
    (col("bp_diastolic").isNotNull()) &
    (col("bp_diastolic") != "")
)

# write to bronze
(
    df_final.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("workspace.ckd_project.ckd_bronze")
    .awaitTermination()
)

# check
display(spark.table("workspace.ckd_project.ckd_bronze"))

bp_diastolic,bp_limit,sg,al,class,rbc,su,pc,pcc,ba,bgr,bu,sod,sc,pot,hemo,pcv,rbcc,wbcc,htn,dm,cad,appet,pe,ane,grf,stage,affected,age
0,0,1.019 - 1.021,1 - 1,ckd,0,< 0,0,0,0,< 112,< 48.1,138 - 143,< 3.65,< 7.31,11.3 - 12.6,33.5 - 37.4,4.46 - 5.05,7360 - 9740,0,0,0,0,0,0,≥ 227.944,s1,1,< 12
0,0,1.009 - 1.011,< 0,ckd,0,< 0,0,0,0,112 - 154,< 48.1,133 - 138,< 3.65,< 7.31,11.3 - 12.6,33.5 - 37.4,4.46 - 5.05,12120 - 14500,0,0,0,0,0,0,≥ 227.944,s1,1,< 12
0,0,1.009 - 1.011,≥ 4,ckd,1,< 0,1,0,1,< 112,48.1 - 86.2,133 - 138,< 3.65,< 7.31,8.7 - 10,29.6 - 33.5,4.46 - 5.05,14500 - 16880,0,0,0,1,0,0,127.281 - 152.446,s1,1,< 12
1,1,1.009 - 1.011,3 - 3,ckd,0,< 0,0,0,0,112 - 154,< 48.1,133 - 138,< 3.65,< 7.31,13.9 - 15.2,41.3 - 45.2,4.46 - 5.05,7360 - 9740,0,0,0,0,0,0,127.281 - 152.446,s1,1,< 12
0,0,1.015 - 1.017,< 0,ckd,0,< 0,0,0,0,154 - 196,< 48.1,133 - 138,< 3.65,< 7.31,13.9 - 15.2,37.4 - 41.3,5.05 - 5.64,7360 - 9740,0,1,0,1,1,0,127.281 - 152.446,s1,1,12 - 20
1,1,≥ 1.023,< 0,notckd,0,< 0,0,0,0,< 112,< 48.1,133 - 138,< 3.65,< 7.31,≥ 16.5,≥ 49.1,5.05 - 5.64,4980 - 7360,0,0,0,0,0,0,102.115 - 127.281,s1,0,12 - 20
0,0,1.019 - 1.021,3 - 3,ckd,0,< 0,0,0,0,< 112,< 48.1,138 - 143,< 3.65,< 7.31,10 - 11.3,29.6 - 33.5,3.28 - 3.87,7360 - 9740,1,1,0,0,0,0,177.612 - 202.778,s1,1,12 - 20
0,0,1.019 - 1.021,< 0,ckd,0,< 0,0,0,0,112 - 154,48.1 - 86.2,133 - 138,< 3.65,< 7.31,11.3 - 12.6,37.4 - 41.3,4.46 - 5.05,4980 - 7360,0,0,0,0,0,0,26.6175 - 51.7832,s4,1,12 - 20
0,0,≥ 1.023,< 0,notckd,0,< 0,0,0,0,112 - 154,48.1 - 86.2,133 - 138,< 3.65,< 7.31,13.9 - 15.2,37.4 - 41.3,5.05 - 5.64,< 4980,0,0,0,0,0,0,26.6175 - 51.7832,s4,0,20 - 27
1,2,1.009 - 1.011,≥ 4,ckd,0,< 0,1,1,1,< 112,< 48.1,123 - 128,< 3.65,< 7.31,7.4 - 8.7,21.8 - 25.7,3.87 - 4.46,12120 - 14500,0,0,0,0,0,1,51.7832 - 76.949,s3,1,20 - 27


In [0]:
print(spark.catalog.currentCatalog())

workspace
